# CSE 3521: Feature Engineering (Group Gamma)

Builds our two text representations, **Bag of Words** and **TF-IDF**, makes the feature visualizations, and tests the contraction-aware tokenizer (M3).

Every notebook reads the same `data/group_cleaned_seed44.csv` and its `split` column, so all of them see exactly the same training and test reviews.

> **M3 update (data pipeline):** this notebook now reads the shared `data/group_cleaned_seed44.csv` built by `scripts/clean_reviews_script_final.py` and uses its `split` column, instead of cleaning and splitting on its own. **Saved outputs below are from the M2 run** (HTML-only cleaning, default tokenizer) until the notebook is re-run with *Run All*.

## 1. Load the shared cleaned data and split

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.naive_bayes import MultinomialNB

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'README.md').is_file() and (p / 'data').is_dir())
sys.path.insert(0, str(ROOT))
FIG_DIR = ROOT / 'report' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

from scripts.preprocessing import (DEFAULT_TOKEN_PATTERN, TOKEN_PATTERN,
                                   load_cleaned, make_vectorizer, split_frames)

df = load_cleaned(ROOT / 'data')
splits = split_frames(df)
X_train, y_train = splits['train']['Text'], splits['train']['Sentiment']
X_test, y_test = splits['test']['Text'], splits['test']['Sentiment']
print('Usable reviews:', len(df), '| train:', len(X_train), '| test:', len(X_test))

## 2. Build both representations

Both use unigrams + bigrams and drop terms that appear in fewer than 2 training reviews. Vectorizers are **fit on the training set only**, so nothing from the test set leaks into the vocabulary or the IDF weights.

* **Bag of Words**: each feature is the raw count of a term in the review.
* **TF-IDF**: count × inverse document frequency, so terms that appear in almost every review (*the*, *this*, *product*) are down-weighted and rarer, more distinctive terms stand out. `sublinear_tf=True` uses 1 + log(count) so a word repeated 10 times does not count 10× as much. Each review vector is L2-normalised, so long and short reviews are on the same scale.

Both use the **contraction-aware token pattern** `TOKEN_PATTERN` from `scripts/preprocessing.py`, so `can't` and `don't` stay single tokens instead of becoming `can` and `don`. Section 7 measures what this changes.

In [ ]:
common = dict(ngram_range=(1, 2), min_df=2, lowercase=True, token_pattern=TOKEN_PATTERN)

# Full vocabulary size before the 50,000 cap, to report in the write-up.
full_vocab = len(CountVectorizer(**common).fit(X_train).vocabulary_)

bow = CountVectorizer(**common, max_features=50000)
X_train_bow = bow.fit_transform(X_train)

tfidf = TfidfVectorizer(**common, max_features=50000, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'Terms with min_df=2 before the cap: {full_vocab:,}')
print(f'BoW vocabulary size:    {len(bow.vocabulary_):,}')
print(f'TF-IDF vocabulary size: {len(tfidf.vocabulary_):,}')
print(f'TF-IDF train matrix: {X_train_tfidf.shape}, non-zero share: '
      f'{X_train_tfidf.nnz / np.prod(X_train_tfidf.shape):.4%}')

## 3. What TF-IDF changes: the same review under both representations

Compare the highest-weighted terms of one review. BoW ranks by raw count, so filler words win; TF-IDF pushes them down.

In [3]:
example = X_train.iloc[0]
print(example[:300], '...\n')

def top_terms(vectorizer, matrix_row, n=8):
    row = matrix_row.toarray().ravel()
    idx = row.argsort()[::-1][:n]
    names = vectorizer.get_feature_names_out()
    return [(names[i], round(float(row[i]), 3)) for i in idx if row[i] > 0]

print('BoW top terms:   ', top_terms(bow, bow.transform([example])))
print('TF-IDF top terms:', top_terms(tfidf, tfidf.transform([example])))

Why are they still selling this stuff if it can kill your pet!?! Buy treats that are made in the U.S. or make them at home yourself. Don't give this trash to your animals. ...

BoW top terms:    [('your', 2.0), ('this', 2.0), ('are', 2.0), ('if', 1.0), ('or', 1.0), ('your pet', 1.0), ('them at', 1.0), ('to your', 1.0)]
TF-IDF top terms: [('your animals', 0.234), ('the or', 0.229), ('stuff if', 0.219), ('why are', 0.214), ('or make', 0.208), ('don give', 0.202), ('selling this', 0.201), ('kill', 0.19)]


## 4. Top-10 most informative TF-IDF features for each sentiment

For every term we take its **average TF-IDF weight in positive training reviews minus its average in negative training reviews**. A large positive gap means the term is typical of positive reviews; a large negative gap means negative reviews. This needs no trained model, so it describes the feature space itself.

In [4]:
terms = tfidf.get_feature_names_out()
is_pos = (y_train == 'positive').to_numpy()
mean_pos = np.asarray(X_train_tfidf[is_pos].mean(axis=0)).ravel()
mean_neg = np.asarray(X_train_tfidf[~is_pos].mean(axis=0)).ravel()
gap = mean_pos - mean_neg

order = np.argsort(gap)
top_pos = pd.DataFrame({'term': terms[order[::-1][:10]], 'gap': gap[order[::-1][:10]]})
top_neg = pd.DataFrame({'term': terms[order[:10]], 'gap': gap[order[:10]]})
print('Top 10 POSITIVE TF-IDF features:'); print(top_pos.round(4).to_string(index=False))
print('\nTop 10 NEGATIVE TF-IDF features:'); print(top_neg.round(4).to_string(index=False))

Top 10 POSITIVE TF-IDF features:
     term    gap
    great 0.0112
     love 0.0084
     best 0.0066
 the best 0.0063
delicious 0.0050
     good 0.0047
  perfect 0.0043
  this is 0.0042
 is great 0.0041
    loves 0.0041

Top 10 NEGATIVE TF-IDF features:
        term     gap
         not -0.0129
         was -0.0109
        were -0.0062
disappointed -0.0060
         bad -0.0059
       would -0.0055
         had -0.0055
       money -0.0053
      it was -0.0050
          be -0.0046


## 5. Figure: most distinctive TF-IDF terms (diverging bar chart)

In [ ]:
n = 15
idx = np.concatenate([order[:n], order[::-1][:n][::-1]])
fig, ax = plt.subplots(figsize=(8, 8))
colors = ['#c0392b' if g < 0 else '#2471a3' for g in gap[idx]]
ax.barh(terms[idx], gap[idx], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean TF-IDF in positive reviews − mean TF-IDF in negative reviews')
ax.set_title('Most distinctive TF-IDF terms (training set)\nred = negative, blue = positive')
fig.tight_layout()
fig.savefig(FIG_DIR / 'm3_tfidf_top_terms.png', dpi=170, bbox_inches='tight')
plt.show()

## 6. Figure: 2-D projection of the TF-IDF feature space

TF-IDF vectors have 50,000 dimensions. **Truncated SVD** (PCA for sparse matrices) keeps the 2 directions with the most variance so we can plot them. We plot a random 4,000-review subset of the training set so the dots stay readable.

In [ ]:
svd = TruncatedSVD(n_components=2, random_state=44)
coords = svd.fit_transform(X_train_tfidf)
print('Variance explained by 2 components:', f'{svd.explained_variance_ratio_.sum():.2%}')

rng = np.random.default_rng(44)
pick = rng.choice(len(coords), size=4000, replace=False)
labels = y_train.to_numpy()[pick]

fig, ax = plt.subplots(figsize=(7, 6))
for label, color in [('positive', '#2471a3'), ('negative', '#c0392b')]:
    m = labels == label
    ax.scatter(coords[pick][m, 0], coords[pick][m, 1], s=6, alpha=0.4, c=color, label=label)
ax.set_xlabel('SVD component 1'); ax.set_ylabel('SVD component 2')
ax.set_title('TF-IDF feature space projected to 2-D (4,000 training reviews)')
ax.legend(markerscale=3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'm3_tfidf_svd_scatter.png', dpi=170, bbox_inches='tight')
plt.show()

## 7. Tokenizer ablation: does keeping contractions help?

Error analysis in M2 showed that scikit-learn's default tokenizer turns *"can't beat"* into *"can beat"*: the pattern `\b\w\w+\b` splits on the apostrophe and drops the one-letter `t`. Our `TOKEN_PATTERN` keeps `can't`, `don't`, `isn't` whole, so bigrams like *"don't like"* survive.

To keep the test set untouched, this comparison trains on the `fit` rows and scores on the **validation slice** of train (`is_val`). Hyperparameters are fixed at the M2 values (Naive Bayes α = 0.5; Logistic Regression C = 1), so only the tokenizer changes. We report macro-F1 and negative-class F1, since negative reviews are the class we care about most.

In [ ]:
fit, val = splits['fit'], splits['val']

def evaluate(name, pattern):
    rows = []
    models = [
        ('Naive Bayes (BoW)', make_vectorizer('bow', token_pattern=pattern), MultinomialNB(alpha=0.5)),
        ('Logistic Regression (TF-IDF)', make_vectorizer('tfidf', token_pattern=pattern),
         LogisticRegression(C=1.0, max_iter=1000, random_state=44)),
    ]
    for model_name, vec, clf in models:
        Xf = vec.fit_transform(fit['Text'])
        pred = clf.fit(Xf, fit['Sentiment']).predict(vec.transform(val['Text']))
        vocab = vec.get_feature_names_out()
        rows.append({
            'Tokenizer': name,
            'Model': model_name,
            'Accuracy': accuracy_score(val['Sentiment'], pred),
            'F1 (macro)': f1_score(val['Sentiment'], pred, average='macro'),
            'Negative F1': f1_score(val['Sentiment'], pred, pos_label='negative'),
            'Vocabulary': len(vocab),
            "Terms with an apostrophe": int(sum("'" in t for t in vocab)),
        })
    return rows

ablation = pd.DataFrame(evaluate('default (splits contractions)', DEFAULT_TOKEN_PATTERN)
                        + evaluate("contraction-aware", TOKEN_PATTERN))
ablation = ablation.sort_values(['Model', 'Tokenizer']).reset_index(drop=True)
print(f"Fit rows: {len(fit):,} | validation rows: {len(val):,}")
print(ablation.round(4).to_string(index=False))

# Save as a Markdown table for the M3 report.
table = ablation.copy()
for col in ['Accuracy', 'F1 (macro)', 'Negative F1']:
    table[col] = table[col].map('{:.4f}'.format)
md_lines = ['| ' + ' | '.join(table.columns) + ' |', '|' + ' --- |' * len(table.columns)]
md_lines += ['| ' + ' | '.join(str(v) for v in row) + ' |' for row in table.itertuples(index=False)]
out = ROOT / 'report' / 'M3_tokenizer_ablation.md'
out.write_text('# M3: Tokenizer ablation (validation slice of train)\n\n' + '\n'.join(md_lines) + '\n',
               encoding='utf-8')
print('Saved:', out)

In [ ]:
# Which negation features exist only with the contraction-aware tokenizer?
vec = make_vectorizer('bow').fit(fit['Text'])
neg_terms = [t for t in vec.get_feature_names_out()
             if t.split()[0] in {"don't", "can't", "isn't", "didn't", "won't", "wasn't", "doesn't"}]
counts = np.asarray(vec.transform(fit['Text'])[:, [vec.vocabulary_[t] for t in neg_terms]].sum(axis=0)).ravel()
top = pd.Series(counts, index=neg_terms).sort_values(ascending=False).head(15)
print('Most frequent negation features (unigrams and bigrams):')
print(top.to_string())

## 8. Features for the deep model

The feedforward network trains on the `fit` rows, tracks validation loss on the `val` rows (for the training/validation loss curves), and is scored once on `test`, the same test set as the classical models. The vectorizer is fit on `fit` only, so neither validation nor test text leaks into the vocabulary or IDF weights.

```python
from scripts.preprocessing import load_cleaned, split_frames, make_vectorizer
splits = split_frames(load_cleaned())
vec = make_vectorizer('tfidf')                    # same settings as this notebook
X_fit = vec.fit_transform(splits['fit']['Text'])
X_val = vec.transform(splits['val']['Text'])
X_test = vec.transform(splits['test']['Text'])
```

For the app, save the vectorizer and network together (for example a scikit-learn `Pipeline` or the vectorizer saved next to the weights) and call `preprocess()` on new text, as `app.py` does.

In [ ]:
vec = make_vectorizer('tfidf')
X_fit = vec.fit_transform(splits['fit']['Text'])
X_val = vec.transform(splits['val']['Text'])
X_test_dl = vec.transform(splits['test']['Text'])
for name, X, y in [('fit', X_fit, splits['fit']['Sentiment']),
                   ('val', X_val, splits['val']['Sentiment']),
                   ('test', X_test_dl, splits['test']['Sentiment'])]:
    print(f"{name:>4}: {X.shape[0]:>6,} rows x {X.shape[1]:,} features | "
          f"negative share {(y == 'negative').mean():.3f}")